# 日本語 BERT モデルによる文書分類

書籍 2-2 に対応し、日本語 BERT モデルをファインチューニングして文書分類します。

下では GPU を用いて深層学習モデルのファインチューニングを行うので、「ランタイム」→「ランタイムのタイプを変更」からハードウェア アクセラレータを `GPU`（T4 など）にしておいてください（CPU のままだと大幅に時間がかかります）。

In [1]:
import torch

if torch.cuda.is_available():
    print("使用可能な GPU:", torch.cuda.get_device_name(0))
else:
    print("警告: GPU が見つかりません（CPU で実行されます）。Colab の「ランタイム」→「ランタイムのタイプを変更」から GPU を選択してください。")

使用可能な GPU: Tesla T4


## 準備

必要なパッケージや辞書をインストールします。

In [2]:
!pip install -q "transformers[ja]" unidic-lite sentencepiece datasets evaluate scikit-learn

In [3]:
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline, set_seed

## 日本語 BERT

日本語 BERT の事前学習モデルは HuggingFace Transformers で公開されているものが一番使いやすいです。

今回は東北大の BERT-base モデル (v3)（`tohoku-nlp/bert-base-japanese-v3`）を使ってみます（以前は cl-tohoku という名前で公開されていましたが、現在は tohoku-nlp という名前に変わっています）。これは日本語の CC-100 および Wikipedia データセットに対して MeCab + UniDic（`unidic-lite`）で形態素解析し、さらに WordPiece アルゴリズムを用いてサブワード化したモデルです。

- https://huggingface.co/tohoku-nlp
- https://github.com/cl-tohoku/bert-japanese

> **補足**: ここでは `AutoTokenizer` / `AutoModelForSequenceClassification` を使用していますが、このモデルでは内部的に `BertJapaneseTokenizer` / `BertForSequenceClassification` が自動的に読み込まれます（直接それらを指定しても同じ結果になります）。`Auto*` クラスを使うと、`model_name` を変えるだけで別の事前学習モデルへ簡単に差し替えることができます。

## マスク言語モデルによる推論

BERT はマスク言語モデルによって事前学習されています。そのため、事前学習モデルを使えば、文中の任意のトークンをマスクしても予測を行うことができます。

In [4]:
model_name = "tohoku-nlp/bert-base-japanese-v3"

tokenizer = AutoTokenizer.from_pretrained(model_name)

fitb = pipeline('fill-mask', model=model_name, tokenizer=tokenizer)
fitb("私は今朝[MASK]を食べました。")

[{'score': 0.11582829803228378,
  'token': 13972,
  'token_str': 'パン',
  'sequence': '私 は 今朝 パン を 食べ まし た 。'},
 {'score': 0.05930883437395096,
  'token': 19487,
  'token_str': 'カレー',
  'sequence': '私 は 今朝 カレー を 食べ まし た 。'},
 {'score': 0.04844663664698601,
  'token': 4828,
  'token_str': '肉',
  'sequence': '私 は 今朝 肉 を 食べ まし た 。'},
 {'score': 0.04084821045398712,
  'token': 6784,
  'token_str': '魚',
  'sequence': '私 は 今朝 魚 を 食べ まし た 。'},
 {'score': 0.03752090409398079,
  'token': 12546,
  'token_str': 'それ',
  'sequence': '私 は 今朝 それ を 食べ まし た 。'}]

## 文書分類のためのファインチューニング

2値分類タスクでファインチューニングを行います。
データは第5章で自ら作成したデータセットを使っていただく想定ですが、サンプルデータを用意しました。新型コロナウイルスに関する体験談風の文書1,000件に、「書き手（またはその家族）が新型コロナウイルスに感染したことが書かれているか」を TRUE/FALSE でラベル付けしたものです（TRUE と FALSE が500件ずつ）。「コメント」の列にラベルを付けた理由を書いてあります。

### データセット

In [5]:
!curl -fLO https://raw.githubusercontent.com/mamorlis/nlpbook/refs/heads/main/misc/covid-annotation.csv

In [6]:
df = pd.read_csv("covid-annotation.csv", usecols=["文書ID", "ラベル", "文書", "コメント"])
df = df.rename(columns={"ラベル": "label", "文書": "text", "コメント": "comment"})
df["label"] = df["label"].astype(int)
print(len(df))
print(df["label"].value_counts())
df.head()

1000
label
0    500
1    500
Name: count, dtype: int64


,文書ID,label,text,comment
0,1,0,幸い、私の場合は軽症で済みましたが、その時から「自分もいつかコロナウイルスに感染する」という...,予防意識・仮定の記述
1,2,0,私の隣室には日本人女性が住んでいますが、彼女も同じ日に発熱して病院に行きました。 彼女と私は...,発熱・接触の可能性のみ
2,3,0,最初は体のだるさだけでしたが、次第に関節痛や筋肉痛なども出るようになり悪化していきました。 ...,ALSの診断（別疾患）
3,4,0,私は、その後も何度か検査を受けて陰性でしたが、今でも時々微熱や咳などの症状が出ます。 このよ...,検査陰性・微熱と咳の相談
4,5,1,今年1月末に体調が急変し、入院して治療と投薬を受けていました。 入院中のことですが、新型コロ...,入院中の2回目検査でコロナ陽性


In [7]:
from datasets import ClassLabel, Dataset
from sklearn.model_selection import train_test_split

# 全体の10%をテストデータ、残りの10%を開発データ、それ以外を訓練データにする
# ch1_tfidf_baseline.ipynb・ch2_svm_knn.ipynb と同じ分割になるよう scikit-learn の train_test_split を使用する
train_dev_df, test_df = train_test_split(df, test_size=0.1, stratify=df["label"], random_state=42)
train_df, dev_df = train_test_split(train_dev_df, test_size=0.1, stratify=train_dev_df["label"], random_state=42)
train_df = train_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(len(train_df), len(dev_df), len(test_df))
print("テスト先頭5件の文書ID:", test_df["文書ID"].head(5).tolist())

label_feature = ClassLabel(names=["FALSE", "TRUE"])
train_raw = Dataset.from_pandas(train_df, preserve_index=False).cast_column("label", label_feature)
dev_raw   = Dataset.from_pandas(dev_df, preserve_index=False).cast_column("label", label_feature)
test_raw  = Dataset.from_pandas(test_df, preserve_index=False).cast_column("label", label_feature)

def encode(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512)

# 文書ID・text・comment 列は訓練には使わないので消しておく（warning を消すため）
remove_cols = ["文書ID", "text", "comment"]
train_dataset = train_raw.map(encode, batched=True, remove_columns=remove_cols)
dev_dataset   = dev_raw.map(encode, batched=True, remove_columns=remove_cols)
test_dataset  = test_raw.map(encode, batched=True, remove_columns=remove_cols)
print(len(train_dataset), len(dev_dataset), len(test_dataset))

810 90 100
テスト先頭5件の文書ID: [789, 861, 668, 200, 288]


810 90 100


### モデル

In [8]:
# id2label を指定しておくと、推論結果に LABEL_0 ではなく FALSE/TRUE と表示される
set_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "FALSE", 1: "TRUE"},
    label2id={"FALSE": 0, "TRUE": 1},
)

### 評価尺度

正解率（accuracy）に加えて、TRUE のクラスについての F1（適合率と再現率の調和平均）も計算します。

In [9]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {**accuracy.compute(predictions=predictions, references=labels),
            **f1.compute(predictions=predictions, references=labels)}

### ハイパーパラメータ

In [11]:
from transformers import TrainingArguments, Trainer
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="test_trainer",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    warmup_steps=0.1,  # 最初の10%のステップで学習率を徐々に上げる
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    save_total_limit=1,
    seed=42,
    report_to="none",  # Weights & Biases などへのログ送信をしない
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.111750,0.966667,0.967033
2,No log,0.061566,0.988889,0.988764
3,No log,0.062380,0.988889,0.988764
4,No log,0.061703,0.988889,0.988764


TrainOutput(global_step=204, training_loss=0.14611664005354338, metrics={'train_runtime': 57.4788, 'train_samples_per_second': 56.369, 'train_steps_per_second': 3.549, 'total_flos': 241283115540720.0, 'train_loss': 0.14611664005354338, 'epoch': 4.0})

In [14]:
# 2026年10月に試したときは、テストデータでの正解率・F1 はどちらも0.95〜0.97程度でした（GPU の計算には毎回わずかな揺らぎがあるので、結果は少し変わります）
trainer.evaluate(eval_dataset=test_dataset)

Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.161081,4,0.960000,0.960000


{'eval_loss': 0.16108126938343048, 'eval_accuracy': 0.96, 'eval_f1': 0.96}

## ファインチューニングしたモデルで推論

ファインチューニングしたモデルを使って、テキストを分類してみます。

In [15]:
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

In [16]:
classifier(
    "私の隣室には日本人女性が住んでいますが、彼女も同じ日に発熱して病院に行きました。 彼女と私は一緒に治療を受けているため、お互いに接触している可能性が高いです。",
    truncation=True,
    max_length=512,
)

[{'label': 'FALSE', 'score': 0.9962345957756042}]

## テストデータの混同行列と誤り分析

`trainer.predict(test_dataset)` を使ってテストデータ全体の予測を取得し、混同行列（confusion matrix）と分類レポートを確認します。
さらに、誤分類された事例を元の「コメント」列と一緒に表示することで、BERT がどのような文脈（否定・仮定・他者の感染など）で誤分類したかを分析できます。このノートブックでは `ch1_tfidf_baseline.ipynb` と同じ分割（`文書ID`）を使っているため、続いて同じテストデータ上で TF-IDF ベースラインとの直接比較も行います。

In [17]:
from sklearn.metrics import confusion_matrix, classification_report

pred_output = trainer.predict(test_dataset)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=-1)

# softmax で予測クラスの確信度を計算する
exp_logits = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
probs = exp_logits / exp_logits.sum(axis=-1, keepdims=True)
confidences = probs[np.arange(len(y_pred)), y_pred]

label_names = ["FALSE", "TRUE"]
cm = pd.DataFrame(
    confusion_matrix(y_true, y_pred),
    index=[f"正解:{name}" for name in label_names],
    columns=[f"予測:{name}" for name in label_names],
)
print("混同行列:")
display(cm)
print("\n分類レポート:")
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

混同行列:


,予測:FALSE,予測:TRUE
正解:FALSE,48,2
正解:TRUE,2,48



分類レポート:
              precision    recall  f1-score   support

       FALSE     0.9600    0.9600    0.9600        50
        TRUE     0.9600    0.9600    0.9600        50

    accuracy                         0.9600       100
   macro avg     0.9600    0.9600    0.9600       100
weighted avg     0.9600    0.9600    0.9600       100



In [18]:
error_df = pd.DataFrame({
    "文書ID": test_df["文書ID"],
    "正解": [label_names[i] for i in y_true],
    "予測": [label_names[i] for i in y_pred],
    "確信度": np.round(confidences, 4),
    "文書": test_df["text"],
    "コメント": test_df["comment"],
})
misclassified = error_df[error_df["正解"] != error_df["予測"]].sort_values("確信度", ascending=False)
print(f"誤分類件数: {len(misclassified)} / {len(error_df)}")
misclassified

誤分類件数: 4 / 100


,文書ID,正解,予測,確信度,文書,コメント
42,37,TRUE,FALSE,0.9940,今回のコロナウイルスによる肺炎という病気は 私には初めてですので、どんな治療法があるのかも分...,コロナウイルスによる肺炎に罹患
48,32,FALSE,TRUE,0.9879,先日、検査結果から新型コロナウイルス陽性反応が出たため、 病院へ行きました。 そこでPCR検...,簡易検査陽性の後のPCR検査で陰性
7,47,TRUE,FALSE,0.9667,今月7日に体調が急変して入院した際には肺炎と診断されました。 先週の検査ではウイルス性肺炎で...,アノテーション元データのラベル準拠
24,31,FALSE,TRUE,0.9524,体調不良により入院することになりました。 入院中には新型コロナウイルスの検査も行われ、陰性の...,入院時のコロナ検査で陰性


### TF-IDF + ロジスティック回帰との直接比較（同一テスト分割）

`ch1_tfidf_baseline.ipynb` と同一の訓練データ（`train_df`）で TF-IDF（文字 1〜3-gram）+ ロジスティック回帰を学習し、同じテストデータ（`test_df`）で日本語 BERT (v3) と比較します。TF-IDF が誤分類し、BERT が正しく分類できた事例（および BERT でも誤った事例）を確認しましょう。

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline

tfidf_clf = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3), min_df=2),
    LogisticRegression(max_iter=1000, C=3.0, random_state=42),
)
tfidf_clf.fit(train_df["text"], train_df["label"])
tfidf_pred = tfidf_clf.predict(test_df["text"])

comparison_df = pd.DataFrame([
    {
        "モデル": "TF-IDF + ロジスティック回帰",
        "Accuracy": accuracy_score(y_true, tfidf_pred),
        "F1": f1_score(y_true, tfidf_pred),
    },
    {
        "モデル": "日本語 BERT (v3)",
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
    },
]).round(4)
display(comparison_df)

compare_cases = pd.DataFrame({
    "文書ID": test_df["文書ID"],
    "正解": [label_names[i] for i in y_true],
    "TF-IDF予測": [label_names[i] for i in tfidf_pred],
    "BERT予測": [label_names[i] for i in y_pred],
    "文書": test_df["text"],
    "コメント": test_df["comment"],
})

tfidf_err_bert_ok = compare_cases[(tfidf_pred != y_true) & (y_pred == y_true)]
print(f"\nTF-IDF が誤り、BERT が正解した事例: {len(tfidf_err_bert_ok)} 件")
display(tfidf_err_bert_ok)

bert_remaining_err = compare_cases[y_pred != y_true]
print(f"\nBERT でも誤分類した残りの事例: {len(bert_remaining_err)} 件")
display(bert_remaining_err)

,モデル,Accuracy,F1
0,TF-IDF + ロジスティック回帰,0.96,0.9592
1,日本語 BERT (v3),0.96,0.9600



TF-IDF が誤り、BERT が正解した事例: 1 件


,文書ID,正解,TF-IDF予測,BERT予測,文書,コメント
62,16,TRUE,FALSE,TRUE,私は熱が出て病院で検査を受けました。 そして、結果は新型コロナウイルスに感染したというもので...,検査でコロナ感染判明



BERT でも誤分類した残りの事例: 4 件


,文書ID,正解,TF-IDF予測,BERT予測,文書,コメント
7,47,TRUE,FALSE,FALSE,今月7日に体調が急変して入院した際には肺炎と診断されました。 先週の検査ではウイルス性肺炎で...,アノテーション元データのラベル準拠
24,31,FALSE,FALSE,TRUE,体調不良により入院することになりました。 入院中には新型コロナウイルスの検査も行われ、陰性の...,入院時のコロナ検査で陰性
42,37,TRUE,FALSE,FALSE,今回のコロナウイルスによる肺炎という病気は 私には初めてですので、どんな治療法があるのかも分...,コロナウイルスによる肺炎に罹患
48,32,FALSE,TRUE,TRUE,先日、検査結果から新型コロナウイルス陽性反応が出たため、 病院へ行きました。 そこでPCR検...,簡易検査陽性の後のPCR検査で陰性
